In [1]:
import cv2
import os
import shutil

In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from collections import Counter
import random

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
import os
import shutil

img_dir    = r"C:\Users\Sandeep\Desktop\Projects\data\RAF-DB\aligned"
label_file = r"C:\Users\Sandeep\Desktop\Projects\data\RAF-DB\list_patition_label.txt"
output_dir = r"C:\Users\Sandeep\Desktop\Projects\data\RAF-DB\Organized"

emotion_map = {
    1: "surprise",
    2: "fear",
    3: "disgust",
    4: "angry",
    5: "sad",
    6: "neutral",
    7: "happy"
}

for split in ["train", "test"]:
    for emotion in emotion_map.values():
        os.makedirs(os.path.join(output_dir, split, emotion), exist_ok=True)

with open(label_file, "r") as f:
    for line in f:
        parts    = line.strip().split()
        filename = parts[0]               # e.g. train_00001.jpg
        label    = int(parts[1])
        emotion  = emotion_map[label]

        # insert _aligned before the extension
        name, ext = os.path.splitext(filename)
        aligned_filename = f"{name}_aligned{ext}"   # train_00001_aligned.jpg

        split = "train" if filename.startswith("train") else "test"

        src = os.path.join(img_dir, aligned_filename)
        dst = os.path.join(output_dir, split, emotion, aligned_filename)

        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f"Missing: {src}")   # flag any that still don't match

print("\nDone — checking distribution:")

for split in ["train", "test"]:
    print(f"\n{split}:")
    for emotion in sorted(emotion_map.values()):
        path  = os.path.join(output_dir, split, emotion)
        count = len(os.listdir(path))
        print(f"  {emotion:12}: {count}")


Done — checking distribution:

train:
  angry       : 4772
  disgust     : 717
  fear        : 281
  happy       : 2524
  neutral     : 705
  sad         : 1982
  surprise    : 1290

test:
  angry       : 1185
  disgust     : 160
  fear        : 74
  happy       : 680
  neutral     : 162
  sad         : 478
  surprise    : 329


In [5]:
# Dataload and transforrm
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dir = r"C:\Users\Sandeep\Desktop\Projects\data\RAF-DB\Organized\train"
test_dir  = r"C:\Users\Sandeep\Desktop\Projects\data\RAF-DB\Organized\test"

# RAF-DB already has a proper train/test split - we'll carve val out of train
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=train_dir, transform=val_transform)

print("Classes:", train_dataset.classes)
print("Total train images:", len(train_dataset))

Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Total train images: 12271


In [6]:
indices = list(range(len(train_dataset)))
random.seed(42)
random.shuffle(indices)

split_size    = int(0.85 * len(indices))   # RAF-DB train is smaller, keep more for training
train_indices = indices[:split_size]
val_indices   = indices[split_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset,   val_indices)

print(f"Train size: {len(train_subset)}")
print(f"Val size:   {len(val_subset)}")

Train size: 10430
Val size:   1841


In [7]:
# get labels only for the train split
train_labels = [train_dataset.targets[i] for i in train_indices]
class_counts = Counter(train_labels)

print("Train class distribution:", class_counts)

# inverse frequency weighting
total = len(train_labels)
class_weights = {cls: total / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)

train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_subset,   batch_size=32, shuffle=False)

# sanity check
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)   # [32, 3, 224, 224]

Train class distribution: Counter({0: 4053, 3: 2150, 5: 1663, 6: 1114, 4: 611, 1: 604, 2: 235})
Batch shape: torch.Size([32, 3, 224, 224])


In [8]:
resnet = models.resnet18(weights="IMAGENET1K_V1")

for name, param in resnet.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, 7)   # 7 classes this time
model     = resnet.to(device)

dummy  = torch.randn(32, 3, 224, 224).to(device)
output = model(dummy)
print(output.shape)   # torch.Size([32, 7])

torch.Size([32, 7])


In [9]:
# class weights for the loss function (same logic as the sampler)
weights_tensor = torch.tensor(
    [class_weights[i] for i in range(7)],
    dtype=torch.float
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = Adam([
    {"params": resnet.layer3.parameters(), "lr": 0.000001},
    {"params": resnet.layer4.parameters(), "lr": 0.00001},
    {"params": resnet.fc.parameters(),     "lr": 0.0001}
])

num_epochs       = 20
patience         = 10
best_val_acc     = 0.0
patience_counter = 0

for epoch in range(num_epochs):

    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_rafdb_resnet18.pth")
        print(f"  ✓ Best model saved (val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_rafdb_resnet18.pth"))
print("Best model loaded")

Epoch [1/20]  Train Loss: 1.6707  Train Acc: 36.27%  |  Val Loss: 1.6555  Val Acc: 34.38%
  ✓ Best model saved (val acc: 34.38%)


: 

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=train_dataset.classes
))

              precision    recall  f1-score   support

       angry       0.96      0.65      0.78       719
     disgust       0.28      0.49      0.36       113
        fear       0.41      0.41      0.41        46
       happy       0.58      0.60      0.59       374
     neutral       0.51      0.61      0.56        94
         sad       0.58      0.69      0.63       319
    surprise       0.58      0.77      0.66       176

    accuracy                           0.64      1841
   macro avg       0.56      0.60      0.57      1841
weighted avg       0.70      0.64      0.65      1841

